In [1]:
import pandas as pd
from datetime import datetime
import numpy as np

from particletracker.fileparser import field_function_from_file
from particletracker import trajectory

In [2]:
''' === Desired current === '''
I = 750

''' === Main DataFrame === '''

fields = pd.DataFrame({
    'I(A)': [991.63, 500],
    'files': ['2017-03-06_BD-004_Model09_Hall_I=991.63A.dat', '2017-03-06_BD-004_Model09_Hall_I=500A.dat'],
    'field': [None]*2
})


for data_file in fields['files']:
    fields['field'] = field_function_from_file(data_file)



''' === Coordinates === '''
df01 = pd.read_csv(fields['files'][0], sep="\t", skiprows=15)
df01 = df01.drop(0)
df01 = df01.apply(pd.to_numeric, errors='coerce')
df01 = df01.drop(df01.columns[[-1]], axis=1)
df01 = df01.dropna()
df01.columns = ['x(m)', 'y(m)', 'z(m)', 'Bx(T)', 'By(T)', 'Bz(T)']


df02 = pd.read_csv(fields['files'][1], sep="\t", skiprows=15)
df02 = df02.drop(0)
df02 = df02.apply(pd.to_numeric, errors='coerce')
df02 = df02.drop(df02.columns[[-1]], axis=1)
df02 = df02.dropna()
df02.columns = ['x(m)', 'y(m)', 'z(m)', 'Bx(T)', 'By(T)', 'Bz(T)']


points_list = []

for index, row in df01.iterrows():
    point = [float(row['x(m)']), float(row['y(m)']), float(row['z(m)'])]
    points_list.append(point)


In [3]:
def intermediate_field_function(fields, I, points_list):
    b_field_1 = fields['field'][0]
    I_1 = float(fields['I(A)'][0])
    b_field_2 = fields['field'][1]
    I_2 = float(fields['I(A)'][1])


    B_list =[]

    for point in points_list:
        Bx_1, By_1, Bz_1 = b_field_1(point)

        Bx_2, By_2, Bz_2 = b_field_2(point)
    
        B_x = Bx_1 + (Bx_2 - Bx_1) * (I - I_1) / (I_2 - I_1)
        B_y = By_1 + (By_2 - By_1) * (I - I_1) / (I_2 - I_1)
        B_z = Bz_1 + (Bz_2 - Bz_1) * (I - I_1) / (I_2 - I_1)

        B_list.append([B_x, B_y, B_z])

    points_list = pd.DataFrame(points_list)
    intermediate_field = pd.DataFrame(B_list)

    final_field = pd.concat([points_list, intermediate_field], axis=1)

    final_field.columns = ['x(m)', 'y(m)', 'z(m)', 'Bx(T)', 'By(T)', 'Bz(T)']

    
        # save data to file
    header = {
    'timestamp': datetime.now().strftime('%Y-%m-%d_%H-%M-%S'),
    'filename': 'intermediate_field',
    '': '',
    'Data columns': 'x(m), y(m), z(m), Bx(T), By(T), Bz(T)',
    '': '',
    '=============== :BEGIN DATA': '==============='
    }
    header_lines = []
    for key, value in header.items():
        if key == '':
            header_lines.append('')
        elif key == 'Data columns':
            header_lines.append(f'{value}')
        else:
            header_lines.append(f'{key}: {value}')
    header_str = '\n'.join(header_lines)
    np.savetxt('intermediate_field.traj', final_field, 
            fmt='%.12e', delimiter=',', 
            header=header_str,
                comments='# ',
                encoding='utf-8')



intermediate_field_function(fields, I, points_list)

In [4]:
''' ==== Particle features ==== '''
m = 939.3      # proton mass (MeV/c²)
q = 1          # proton charge (elementary charge)



''' === Initial conditions === '''
x0, y0, z0 = 0, 0, 0   # initial coordinates (m)
v_direction = [1, 0, 0]    # velocity vector
Ec0 = 1                    # initial particle's kinetic energy (J)



''' === Fields === '''
e_field = lambda ponto: [0, 0, 0]       # force electric field to be 0 at all points
b_field = field_function_from_file('intermediate_field.traj')


''' === Trajectory solution === '''
method = 'boris'            # choose between "boris" and "rk4"
step = 1e-12                # time step (s)
total_t = 1e-7              # total elapsed time (s)
traj_distance = 0.1         # total distance of particle's trajectory in X axis (m)



# run the simulation
traj = trajectory(m, q, 
                 x0, y0, z0, v_direction, Ec0,
                 b_field, e_field,
                 method, step, total_t, 
                 output_file='traj_calculation_results')

ValueError: Length mismatch: Expected axis has 1 elements, new values have 6 elements

In [ ]:
traj